In [1]:
import re
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))
data = pd.read_csv(file_name)

Saving medium_data.csv to medium_data.csv


In [2]:
TEXT_COLUMN = "title"
texts = (
    data[TEXT_COLUMN]
    .dropna()
    .astype(str)
    .tolist()
)

In [3]:
texts = texts[:5000]
words_in_data = []

for sentence in texts:
  words_in_data.extend(
      re.findall(r"\b[a-zA-Z]+\b",sentence.lower())

  )

In [4]:
counts = Counter(words_in_data)
vocab = ["<UNK>"] + [word for word,
                     count in counts.most_common(2999)]

In [5]:
word_to_id = {word: i for i,word in enumerate(vocab)}
id_to_word = {i: word for i,word in enumerate(vocab)}
print(id_to_word)
print(word_to_id)


{0: '<UNK>', 1: 'the', 2: 'to', 3: 'strong', 4: 'a', 5: 'of', 6: 'how', 7: 'and', 8: 'in', 9: 'your', 10: 'you', 11: 'markup', 12: 'for', 13: 'with', 14: 'is', 15: 's', 16: 'data', 17: 'why', 18: 'i', 19: 'class', 20: 'what', 21: 'on', 22: 'learning', 23: 'from', 24: 'an', 25: 'it', 26: 't', 27: 'be', 28: 'can', 29: 'are', 30: 'writing', 31: 'my', 32: 'design', 33: 'using', 34: 'about', 35: 'machine', 36: 'ux', 37: 'do', 38: 'not', 39: 'python', 40: 'that', 41: 'we', 42: 'science', 43: 'ai', 44: 'life', 45: 'should', 46: 'when', 47: 'need', 48: 'make', 49: 'time', 50: 'more', 51: 'as', 52: 'part', 53: 'business', 54: 'new', 55: 'at', 56: 'or', 57: 'have', 58: 'don', 59: 'get', 60: 'work', 61: 'up', 62: 'will', 63: 'by', 64: 'ways', 65: 'write', 66: 'guide', 67: 'use', 68: 'marketing', 69: 'first', 70: 'know', 71: 'good', 72: 'best', 73: 'deep', 74: 'this', 75: 'analysis', 76: 'better', 77: 'one', 78: 'things', 79: 'if', 80: 'em', 81: 're', 82: 'out', 83: 'writer', 84: 'into', 85: 'me',

In [6]:
X,Y = [],[]
print(X)

[]


In [7]:
for sentence in texts:
  tokens = re.findall(r"\b[a-zA-Z]+\b",sentence.lower())

  ids = [word_to_id.get(word,0)
  for word in tokens
         ]
  for i in range(len(ids)-1):
      if ids[i] != 0 and ids[i +1] !=0:
        X.append(ids[i])
        Y.append(ids[i+1])
X = torch.tensor(X[:20000],dtype=torch.long)
Y = torch.tensor(Y[:20000],dtype=torch.long)
print("Vocabulary size:", len(vocab))
print("Training examples:", len(X))


Vocabulary size: 3000
Training examples: 20000


In [9]:

# ==========================================
# STEP 1: Import required libraries
# ==========================================
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import pandas as pd
from google.colab import files

# ==========================================
# STEP 2: Prepare the dataset
# ==========================================
dataset = TensorDataset(X, Y)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True
)

test_loader = DataLoader(
    test_dataset, batch_size=128, shuffle=False
)

print("Training samples:", train_size)
print("Testing samples:", test_size)

# ==========================================
# STEP 3: Build the neural network
# ==========================================
class NextWordModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim
        )

        self.output_layer = nn.Linear(
            embedding_dim, vocab_size
        )

    def forward(self, x):
        embedded = self.embedding(x)
        output = self.output_layer(embedded)
        return output


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = NextWordModel(
    vocab_size=len(vocab),
    embedding_dim=64
).to(device)

print(model)
print("Device:", device)

# ==========================================
# STEP 4: Define loss and optimizer
# ==========================================
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001
)

# ==========================================
# STEP 5: Train the model
# ==========================================
epochs = 50

for epoch in range(epochs):
    model.train()

    total_loss = 0.0

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Forward propagation
        outputs = model(inputs)

        # Calculate loss
        loss = criterion(outputs, targets)

        # Clear previous gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)

    average_loss = total_loss / train_size

    print(
        f"Epoch [{epoch + 1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )


Training samples: 16000
Testing samples: 4000
NextWordModel(
  (embedding): Embedding(3000, 64)
  (output_layer): Linear(in_features=64, out_features=3000, bias=True)
)
Device: cpu
Epoch [1/50], Loss: 7.8001
Epoch [2/50], Loss: 7.0097
Epoch [3/50], Loss: 6.4996
Epoch [4/50], Loss: 6.1113
Epoch [5/50], Loss: 5.7833
Epoch [6/50], Loss: 5.5017
Epoch [7/50], Loss: 5.2569
Epoch [8/50], Loss: 5.0408
Epoch [9/50], Loss: 4.8485
Epoch [10/50], Loss: 4.6764
Epoch [11/50], Loss: 4.5215
Epoch [12/50], Loss: 4.3820
Epoch [13/50], Loss: 4.2562
Epoch [14/50], Loss: 4.1404
Epoch [15/50], Loss: 4.0359
Epoch [16/50], Loss: 3.9404
Epoch [17/50], Loss: 3.8532
Epoch [18/50], Loss: 3.7735
Epoch [19/50], Loss: 3.7000
Epoch [20/50], Loss: 3.6331
Epoch [21/50], Loss: 3.5714
Epoch [22/50], Loss: 3.5148
Epoch [23/50], Loss: 3.4625
Epoch [24/50], Loss: 3.4140
Epoch [25/50], Loss: 3.3685
Epoch [26/50], Loss: 3.3268
Epoch [27/50], Loss: 3.2888
Epoch [28/50], Loss: 3.2528
Epoch [29/50], Loss: 3.2194
Epoch [30/50], L

In [10]:

# ==========================================
# STEP 6: Evaluate the trained model
# ==========================================
model.eval()

correct_top1 = 0
correct_top5 = 0
total = 0

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model(inputs)

        # Top-1 prediction
        predicted = outputs.argmax(dim=1)

        correct_top1 += (predicted == targets).sum().item()

        # Top-5 predictions
        k = min(5, len(vocab))
        top5 = outputs.topk(k, dim=1).indices

        correct_top5 += (
            top5 == targets.unsqueeze(1)
        ).any(dim=1).sum().item()

        total += targets.size(0)

top1_accuracy = 100 * correct_top1 / total
top5_accuracy = 100 * correct_top5 / total

print(f"Top-1 Accuracy: {top1_accuracy:.2f}%")
print(f"Top-5 Accuracy: {top5_accuracy:.2f}%")


Top-1 Accuracy: 17.88%
Top-5 Accuracy: 30.65%


In [11]:

# ==========================================
# STEP 7: Generate predictions
# ==========================================
model.eval()

with torch.no_grad():
    all_inputs = X.to(device)
    outputs = model(all_inputs)

    predicted_ids = outputs.argmax(dim=1).cpu().tolist()

input_ids = X.tolist()
actual_ids = Y.tolist()

results = pd.DataFrame({
    "Input_Word": [
        id_to_word[int(i)] for i in input_ids
    ],
    "Actual_Next_Word": [
        id_to_word[int(i)] for i in actual_ids
    ],
    "Predicted_Next_Word": [
        id_to_word[int(i)] for i in predicted_ids
    ]
})

results["Correct"] = (
    results["Actual_Next_Word"]
    == results["Predicted_Next_Word"]
)

print("\nSample predictions:")
print(results.head(20).to_string(index=False))

print("\nTotal predictions:", len(results))
print("Correct predictions:", results["Correct"].sum())
print(
    "Prediction accuracy: "
    f"{results['Correct'].mean() * 100:.2f}%"
)



Sample predictions:
Input_Word Actual_Next_Word Predicted_Next_Word  Correct
         a         beginner                  ux    False
  beginner                s                   s     True
         s            guide                 the    False
     guide               to                  to     True
        to             word                  be    False
      word        embedding                 for    False
 embedding             with                  of    False
      with           gensim              python    False
    gensim            model               model     True
     hands               on                  on     True
        on            graph                 the    False
     graph           neural              neural     True
    neural         networks            networks     True
  networks             with                 for    False
      with          pytorch              python    False
   pytorch          pytorch           geometric    False
   pytorch

In [12]:

# ==========================================
# STEP 8: Export predictions to CSV
# ==========================================
output_file = "next_word_predictions.csv"

results.to_csv(output_file, index=False)

print("Predictions saved successfully!")

files.download(output_file)


Predictions saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:

# ==========================================
# STEP 9: Predict the next word interactively
# ==========================================
def predict_next_word(word):
    word = word.lower()

    if word not in word_to_id or word_to_id[word] == 0:
        print("Word not found in the vocabulary.")
        return

    input_id = torch.tensor(
        [word_to_id[word]],
        dtype=torch.long,
        device=device
    )

    model.eval()

    with torch.no_grad():
        output = model(input_id)
        predicted_id = output.argmax(dim=1).item()

    print("Input word:", word)
    print("Predicted next word:", id_to_word[predicted_id])


predict_next_word("machine")
predict_next_word("data")
predict_next_word("learning")


Input word: machine
Predicted next word: learning
Input word: data
Predicted next word: science
Input word: learning
Predicted next word: for


In [15]:

# Runtime user input for next-word prediction

def predict_next_word(word):
    word = word.lower().strip()

    if word not in word_to_id or word_to_id[word] == 0:
        print("Word not found in vocabulary. Try another word.")
        return

    input_id = torch.tensor(
        [word_to_id[word]],
        dtype=torch.long
    ).to(device)

    model.eval()

    with torch.no_grad():
        output = model(input_id)
        predicted_id = output.argmax(dim=1).item()

    print("Input word:", word)
    print("Predicted next word:", id_to_word[predicted_id])


# Get input while the program is running
user_word = input("Enter a word: ")

predict_next_word(user_word)


Enter a word: right
Input word: right
Predicted next word: way


In [16]:

while True:
    user_word = input(
        "\nEnter a word (or type 'exit' to stop): "
    ).strip()

    if user_word.lower() == "exit":
        print("Program stopped. Thank you!")
        break

    if not user_word:
        print("Please enter a word.")
        continue

    predict_next_word(user_word)



Enter a word (or type 'exit' to stop): wrong
Input word: wrong
Predicted next word: with

Enter a word (or type 'exit' to stop): stop
Input word: stop
Predicted next word: trying

Enter a word (or type 'exit' to stop): exit
Program stopped. Thank you!
